In [ ]:
import pandas as pd
from django.core.exceptions import ObjectDoesNotExist
from csae_pyutils import gsheet_to_df
from tqdm import tqdm
from apis_core.apis_metainfo.models import Collection, TempEntityClass, Uri
from apis_core.apis_entities.models import Place, Event, Work, Person
from apis_core.apis_vocabularies.models import PlaceEventRelation, EventType, WorkType, PersonWorkRelation
from apis_core.apis_relations.models import PlaceEvent, PersonWork

In [ ]:
col = Collection.objects.filter(name__startswith="Brahms Online").first()
domain = "brahms-online"
place_event_relation = PlaceEventRelation.objects.get(name="Veranstaltungsort von")
col, place_event_relation

In [ ]:
df = gsheet_to_df("1QV0F8XR8fbbWPNAbdj0TWEkRkSzuwW06Ox_o1gvEOGg")

In [ ]:
df.head(3)

In [ ]:
for i, row in tqdm(df.iterrows(), total=len(df)):
    tmp_id = row["URI(s)"].strip()
    pmb_uri = f"https://brahms-online.oeaw.ac.at/{tmp_id}.html"
    event = Event.objects.get_or_create(name=row["titel"])[0]
    entity = TempEntityClass.objects.get(id=event.id)
    pmb_uri, _ = Uri.objects.get_or_create(uri=pmb_uri, domain=domain, entity=entity)
    entity.collection.add(col)
    event_type = EventType.objects.get(id=row["type"])
    event.kind = event_type
    start_date_written = row["writtenDate"] if pd.isna(row["when-iso"]) else row["when-iso"]
    event.start_date_written = start_date_written
    event.save()
    try:
        related_place = Place.objects.get(id=int(row["venue"]))
    except (ObjectDoesNotExist, ValueError):
        related_place = Place.objects.filter(name=row["venue-name"]).first()
    if related_place is None:
        try:
            related_place = Place.objects.get_or_create(name=row["place"])[0]
        except:
            related_place = Place.objects.filter(name=row["place"]).last()
    PlaceEvent.objects.get_or_create(related_place=related_place, related_event=event, relation_type=place_event_relation, start_date_written=start_date_written, end_date_written=start_date_written)


In [ ]:
# werke
df = gsheet_to_df("1PklkDTva4jiXH4xsix2CwD-mIk6VrEWT2R56_F_3nMw")
work_type = WorkType.objects.get(id=1072)

In [ ]:
df = df[
    (df["ebene"] == "Werk")
    & (df["komponist"] == "Johannes Brahms")
    & (df["URI(s)"].str.startswith("JBWV_0"))
    & (df["titel"].notna())
]
len(df)
person_work_type = PersonWorkRelation.objects.get(name="hat geschaffen")


In [ ]:
len(df)
person = Person.objects.get(id=10989)
base_url = "https://brahms-online.oeaw.ac.at/"
for i, row in df.iterrows():
    title = row["titel"]
    id = row["URI(s)"].split(";")[0].strip()
    op = row["nummer"]
    title = row["titel"]
    work_name = f"{title}, {op}"
    uri = f"{base_url}{id} - {op} - {title}.html"
    pmb_uri = Uri.objects.get_or_create(uri=uri, domain=domain)[0]
    entity = pmb_uri.entity
    if entity:
        entity.collection.add(col)
        pass
    else:
        item = {
            "name": title[:254],
        }
        entity = Work.objects.create(**item)
    entity.save()
    entity.collection.add(col)
    pmb_uri.entity = entity
    pmb_uri.save()
    work = Work.objects.get(id=entity.id)
    PersonWork.objects.get_or_create(related_person=person, related_work=work, relation_type=person_work_type)
